# Fraud Detection: Advanced Feature Engineering

This notebook focuses on transforming raw transaction data into high-impact predictive signals. We will implement:
1. **Temporal Features**: Time-based patterns.
2. **IP to Country Mapping**: Geographical signals.
3. **Velocity Features**: Frequency and volume per device/IP.
4. **Outlier Flags**: Behavioral anomalies.
5. **Target Encoding**: Safe historical fraud rates for categorical data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import KFold

# Settings
pd.set_option('display.max_columns', None)
tqdm.pandas()

## 1. Data Ingestion
Loading the raw datasets and converting timestamps.

In [2]:
df = pd.read_csv("data/raw/Fraud_Data.csv")
ip_df = pd.read_csv("data/raw/IpAddress_to_Country.csv")

# Convert times to datetime
df['signup_time'] = pd.to_datetime(df['signup_time'])
df['purchase_time'] = pd.to_datetime(df['purchase_time'])

print(f"Transactions loaded: {len(df):,}")
print(f"IP Ranges loaded: {len(ip_df):,}")
df.head()

Transactions loaded: 151,112
IP Ranges loaded: 138,846


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


## 2. Temporal Features
Capturing signals based on when the transaction happened.

In [3]:
def create_temporal_features(df):
    df = df.copy()
    
    # Time between signup and purchase (very important for 'flash' fraud)
    df['account_age_minutes'] = (df['purchase_time'] - df['signup_time']).dt.total_seconds() / 60
    
    # Transaction hour and day
    df['purchase_hour'] = df['purchase_time'].dt.hour
    df['purchase_day_of_week'] = df['purchase_time'].dt.dayofweek
    
    # Binary flags for night and weekend
    df['is_weekend'] = (df['purchase_day_of_week'] >= 5).astype(int)
    df['is_night'] = ((df['purchase_hour'] >= 22) | (df['purchase_hour'] <= 6)).astype(int)
    
    return df

df = create_temporal_features(df)
df[['account_age_minutes', 'is_night', 'is_weekend']].head()

,account_age_minutes,is_night,is_weekend
0,75111.366667,1,1
1,299.066667,1,0
2,0.016667,0,0
3,8201.416667,0,0
4,72691.016667,0,0


## 3. IP to Country Mapping
Using IP ranges to determine the origin country. We use `pd.merge_asof` for high performance range lookups.

In [4]:
# Normalize IP types
df['ip_address_int'] = df['ip_address'].astype(float)
ip_df = ip_df.sort_values('lower_bound_ip_address')

def map_ip_to_country(df, ip_ranges):
    # Fast range-based merge
    df_merged = pd.merge_asof(
        df.sort_values('ip_address_int'), 
        ip_ranges, 
        left_on='ip_address_int', 
        right_on='lower_bound_ip_address'
    )
    
    # Verify that IP is actually within the upper bound
    df_merged['country'] = np.where(
        df_merged['ip_address_int'] <= df_merged['upper_bound_ip_address'],
        df_merged['country'],
        'Unknown'
    )
    
    return df_merged

df = map_ip_to_country(df, ip_df)
df['country'].value_counts().head(10)

country
United States        58049
Unknown              21966
China                12038
Japan                 7306
United Kingdom        4490
Korea Republic of     4162
Germany               3646
France                3161
Canada                2975
Brazil                2961
Name: count, dtype: int64

## 4. Velocity Features
Calculating counts of transactions per device within a 24-hour window. This helps detect bots.

In [5]:
def create_velocity_features(df):
    # Sort by time to prevent data leakage in rolling calculations
    df = df.sort_values('purchase_time')
    
    # Transactions per device in the last 24 hours
    # Note: We use a simplified frequency count here for performance
    df['device_txn_count_total'] = df.groupby('device_id')['user_id'].transform('count')
    
    # Flag for first-time devices
    df['is_device_new'] = (df.groupby('device_id')['purchase_time'].transform('rank') == 1).astype(int)
    
    return df

df = create_velocity_features(df)
df[['device_id', 'device_txn_count_total', 'is_device_new']].head()

,device_id,device_txn_count_total,is_device_new
69937,BBPACGBUVJUXF,10,1
69936,BBPACGBUVJUXF,10,0
69929,BBPACGBUVJUXF,10,0
69930,BBPACGBUVJUXF,10,0
69935,BBPACGBUVJUXF,10,0


## 5. Outlier Detection
Creating flags for suspicious transaction amounts.

In [6]:
def create_amount_features(df):
    # Log transform to normalize distribution
    df['purchase_value_log'] = np.log1p(df['purchase_value'])
    
    # Global Z-Score for detecting excessive amounts
    mean_val = df['purchase_value'].mean()
    std_val = df['purchase_value'].std()
    df['purchase_value_zscore'] = (df['purchase_value'] - mean_val) / std_val
    
    # Binary flag for extreme outliers (Z > 3)
    df['is_outlier_amount'] = (df['purchase_value_zscore'].abs() > 3).astype(int)
    
    return df

df = create_amount_features(df)
df[['purchase_value', 'is_outlier_amount']].head()

,purchase_value,is_outlier_amount
69937,14,0
69936,14,0
69929,14,0
69930,14,0
69935,14,0


## 6. Target Encoding
Encoding categorical variables securely using K-Fold statistics to avoid leakage.

In [7]:
def target_encode_kfold(df, col, target='class', n_folds=5, smoothing=10):
    df = df.copy()
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    encoded_col = f"{col}_fraud_rate"
    df[encoded_col] = 0.0
    
    global_mean = df[target].mean()
    
    for train_idx, val_idx in kf.split(df):
        train_data = df.iloc[train_idx]
        
        # Groupby encoding with smoothing
        agg = train_data.groupby(col)[target].agg(['count', 'mean'])
        counts = agg['count']
        means = agg['mean']
        
        smooth_encoding = (counts * means + smoothing * global_mean) / (counts + smoothing)
        df.loc[df.index[val_idx], encoded_col] = df.loc[df.index[val_idx], col].map(smooth_encoding)
        
    df[encoded_col] = df[encoded_col].fillna(global_mean)
    return df

for col in ['browser', 'source', 'country']:
    df = target_encode_kfold(df, col)

df[['country', 'country_fraud_rate']].head()

,country,country_fraud_rate
69937,Korea Republic of,0.087107
69936,Korea Republic of,0.093508
69929,Korea Republic of,0.092197
69930,Korea Republic of,0.094155
69935,Korea Republic of,0.087107


## 7. Cleanup & Export
Dropping unnecessary columns and saving the final curated dataset.

In [8]:
drop_cols = [
    'user_id', 'signup_time', 'purchase_time', 'device_id', 
    'ip_address', 'ip_address_int', 'lower_bound_ip_address', 'upper_bound_ip_address',
    'browser', 'source', 'country', 'sex'
]

final_df = df.drop(columns=drop_cols, errors='ignore')

print(f"Final features: {list(final_df.columns)}")
print(f"Final shape: {final_df.shape}")

Final features: ['purchase_value', 'age', 'class', 'account_age_minutes', 'purchase_hour', 'purchase_day_of_week', 'is_weekend', 'is_night', 'device_txn_count_total', 'is_device_new', 'purchase_value_log', 'purchase_value_zscore', 'is_outlier_amount', 'browser_fraud_rate', 'source_fraud_rate', 'country_fraud_rate']
Final shape: (151112, 16)


### Saving the data

In [9]:
final_df.to_csv("data/processed/feature_engineered.csv", index=False)
print("Success! Curated dataset saved to csv.")

Success! Curated dataset saved to csv.
